# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

First of all, import all the necessary packages, classes and helper fulctions. 

In [112]:
from langchain_community.document_loaders import PyPDFLoader
from dotenv import load_dotenv
from openai import OpenAI
import os
from pydantic import BaseModel, Field
from deepeval import evaluate
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from deepeval.metrics import SummarizationMetric, GEval
from deepeval.models import GPTModel

In [113]:
# create StructuredSummary class for structured output
class StructuredSummary(BaseModel):
    author: str
    title: str
    relevance: str=Field(description="a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.")
    summary: str=Field(description="a concise and succinct summary no longer than 1000 tokens.")
    tone: str=Field(description="the tone used to produce the summary.")
    inputTokens: str=Field(description="number of input tokens (obtain this from the response object).")
    outputTokens: str=Field(description="number of tokens in output (obtain this from the response object).")


In [114]:
# create StructuredEvaluation class for structured output
class StructuredEvaluation(BaseModel):
    SummarizationScore: float
    SummarizationReason: str
    CoherenceScore: float
    CoherenceReason: str
    TonalityScore: float
    TonalityReason: str
    SafetyScore: float
    SafetyReason: str


In [115]:
# define a function that runs all of the evaluation
def my_ultimate_evaluation_set(model, test_case, question_bank, verbose_mode=False):
    """A function that returns the evaluation results of summarization, coherence, tonality and safety of a model summarization. 

    Args:
        model (GPTModel): a LLM model
        test_case (LLMTestCase): a case to be tested
        question_bank (dict of list): assessment questions and evaluation steps
        verbose_mode (bool, optional): if True, output verbose logs. Defaults to False.

    Returns:
        BaseModel: a structured output
    """
    summarization = SummarizationMetric(
        assessment_questions=question_bank['summarization'],
        model=model,
        verbose_mode=verbose_mode
    )

    coherence = GEval(
        name="Coherence",
        evaluation_steps=question_bank['coherence'],
        model=model,
        evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
        verbose_mode=verbose_mode
    )

    tonality = GEval(
        name="Tonality",
        evaluation_steps=question_bank['professionalism'],
        model=model,
        evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
        verbose_mode=verbose_mode
    )

    safety = GEval(
        name="Safety",
        evaluation_steps=question_bank['pii_and_language'],
        model=model,
        evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
        verbose_mode=verbose_mode
    )

    summarization.measure(test_case)
    coherence.measure(test_case)
    tonality.measure(test_case)
    safety.measure(test_case)

    result = StructuredEvaluation(
        SummarizationScore=round(summarization.score,2),
        SummarizationReason=summarization.reason,
        CoherenceScore=round(coherence.score, 2),
        CoherenceReason=coherence.reason,
        TonalityScore=round(tonality.score, 2),
        TonalityReason=tonality.reason,
        SafetyScore=round(safety.score, 2),
        SafetyReason=safety.reason
    )

    questions = {
        'summarization': summarization.assessment_questions, 
        'coherence': coherence.evaluation_steps, 
        'tonality': tonality.evaluation_steps, 
        'safety': safety.evaluation_steps
    }

    return result, questions



In [116]:
question_bank = {
    'summarization': [
        'Is the article about managing oneself?',
        'Does the article emphasize the importance of self-knowledge?',
        'Is it stated that individuals must take responsibility for their own careers?',
        "Does the article suggest that understanding one's strengths is crucial for success?",
        'Is the author Peter F. Drucker?'
    ],
    'coherence': [
        "Evaluate whether the response uses clear and direct language.",
        "Check if the explanation avoids jargon or explains it when used.",
        "Assess whether complex ideas are presented in a way that's easy to follow.",
        "Identify any vague or confusing parts that reduce understanding.",
        "Check if the main idea is consistent with the original document. "
    ],
    'professionalism': [
        "Determine whether the actual output maintains a professional tone throughout.",
        "Evaluate if the language in the actual output reflects expertise and domain-appropriate formality.",
        "Ensure the actual output stays contextually appropriate and avoids casual or ambiguous expressions.",
        "Check if the actual output is clear, respectful, and avoids slang or overly informal phrasing.",
        "Check if each argument in the actual output can be backed by the original document"
    ],
    'pii_and_language': [
        "Check whether the output includes any real or plausible personal information (e.g., names, phone numbers, emails).",
        "Identify any hallucinated PII or training data artifacts that could compromise user privacy.",
        "Ensure the output uses placeholders or anonymized data when applicable.",
        "Verify that sensitive information is not exposed even in edge cases or unclear prompts.",
        "Make sure gender-neutral language is used throughout the entire actual output"
    ]
}

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [117]:
%load_ext dotenv
%dotenv ../05_src/.env
%dotenv ../05_src/.secrets

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [118]:
# Load all pages
file_path = "../07_data/Managing Oneself_Drucker_HBR.pdf"
loader = PyPDFLoader(file_path)

docs = loader.load()

# Merge all pages into one document
document_text = ''

for page in docs:
    document_text += page.page_content + '\n'

## Generation Task

Using the OpenAI SDK, please create a **structured output** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [119]:
# instantiate OpenAI object
client = OpenAI(base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1', 
                api_key='any value',
                default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')})

# dynamically define prompts
user_prompt = f'''
    Pleaes create a summary of this article below:
    <article> 
    {document_text} 
    </article>
'''

my_tone = 'Formal Academic Writing'
sys_prompt = f'''
    Please complete the output in {my_tone} tone
'''

# get response from OpenAI object
response = client.responses.parse(
    model="gpt-4o-mini",
    instructions=sys_prompt,
    temperature=0,
    input=[
        {
            "role": "user",
            "content": user_prompt,
        },
    ],
    text_format=StructuredSummary,
)

my_summary = response.output_parsed
print(my_summary.summary)

In "Managing Oneself," Peter F. Drucker argues that success in the knowledge economy hinges on self-awareness, where individuals must act as their own chief executive officers. He posits that understanding one's strengths, weaknesses, values, and preferred work styles is essential for career development and fulfillment. Drucker introduces feedback analysis as a method for identifying strengths by comparing expected outcomes with actual results, encouraging individuals to focus on enhancing their strengths rather than improving weaknesses. He emphasizes the importance of aligning personal values with organizational values to avoid frustration and underperformance. Additionally, Drucker discusses the necessity of understanding how one learns and performs, advocating for effective communication and relationship management in the workplace. He concludes by addressing the need for individuals to prepare for the second half of their careers, suggesting that proactive engagement in new opport

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [120]:
# create the LLM judge
my_model = GPTModel(
    model="gpt-4o-mini",
    temperature=0,
    # api_key='any value',
    default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')},
    base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
)

# create the test case
my_test_case = LLMTestCase(
    input=user_prompt, 
    actual_output=my_summary.summary
)

# evaluate the test case based on the selected metrics
my_eval_result, my_eval_questions = my_ultimate_evaluation_set(my_model, my_test_case, question_bank=question_bank, verbose_mode=False)
my_eval_str = ''

# format and print the evaluation recommendation
for key in my_eval_result.model_dump().keys():
    my_eval_str += key + ': ' + str(my_eval_result.model_dump()[key]) + '\n'

print(my_eval_str)

Output()

Output()

Output()

Output()

SummarizationScore: 0.8
SummarizationReason: The score is 0.80 because the summary includes a contradiction regarding Drucker's focus on career preparation versus meaningful contributions, and it adds extra information about proactive engagement that is not present in the original text.
CoherenceScore: 0.89
CoherenceReason: The response uses clear and direct language, effectively summarizing Drucker's key concepts without jargon. Complex ideas, such as self-awareness and feedback analysis, are presented in an accessible manner. There are no vague parts, and the main ideas align well with the original document's themes of self-management and career development.
TonalityScore: 0.95
TonalityReason: The response maintains a professional tone and reflects expertise in discussing Drucker's concepts. The language is formal and contextually appropriate, avoiding casual expressions. Each argument presented is clear and aligns with the original document's themes, such as self-awareness and feedb

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [121]:
# We feed the original text, summary and evaluation to the model and ask for specific enhancements. 
# dynamically define prompts
sys_prompt2 = f'''
    Please complete the output in {my_tone} tone. Please address the enhancement recommendations outlined in this evaluation statement:
    <statement>
    <{my_eval_str}>
    </statement>
'''

# get response from OpenAI object
response2 = client.responses.parse(
    model="gpt-4o-mini",
    temperature=0,
    instructions=sys_prompt2,
    input=[
        {
            "role": "user",
            "content": user_prompt,
        },
    ],
    text_format=StructuredSummary,
)

my_summary2 = response2.output_parsed
print(my_summary2.summary)

In "Managing Oneself," Peter F. Drucker argues that success in the knowledge economy hinges on self-awareness and personal responsibility. He posits that individuals must act as their own chief executive officers, taking charge of their careers in an era where companies no longer manage employees' paths. To thrive, one must understand their strengths, weaknesses, learning styles, and values. Drucker introduces feedback analysis as a method for identifying strengths by comparing expected outcomes with actual results. He emphasizes the importance of aligning personal values with organizational values to avoid frustration and enhance performance. Additionally, he discusses the necessity of adapting to different work environments and the significance of effective communication and relationship management in achieving results. Drucker concludes by highlighting the need for individuals to prepare for the second half of their careers, suggesting that proactive engagement in new opportunities 

In [122]:
# create the new test case
my_test_case2 = LLMTestCase(
    input=user_prompt, 
    actual_output=my_summary2.summary
)

# re-evaluate the test case based on the selected metrics
my_eval_result2, my_eval_questions2 = my_ultimate_evaluation_set(my_model, my_test_case, question_bank=question_bank, verbose_mode=False)
my_eval_str2 = ''

# format and print the evaluation recommendation
for key in my_eval_result2.model_dump().keys():
    my_eval_str2 += key + ': ' + str(my_eval_result2.model_dump()[key]) + '\n'

print(my_eval_str2)

Output()

Output()

Output()

Output()

SummarizationScore: 0.9
SummarizationReason: The score is 0.90 because the summary effectively captures the main ideas of the original text, with no contradictions present. However, it introduces extra information about proactive engagement in new opportunities, which, while relevant, was not part of the original text. This slight deviation from the original content affects the overall fidelity of the summary.
CoherenceScore: 0.9
CoherenceReason: The response uses clear and direct language, effectively summarizing Drucker's key concepts without jargon. Complex ideas, such as self-awareness and feedback analysis, are presented in an accessible manner. There are no vague parts, and the main ideas align well with the original document's themes of self-management and career development.
TonalityScore: 0.93
TonalityReason: The response maintains a professional tone and reflects expertise in discussing Drucker's concepts. The language is formal and contextually appropriate, avoiding casual e

After the auto-correction, our model reaches a higher score in summarization (0.1), while reaches a slightly lower score for tonality (-0.02). This is due to that even though the client doesn't store any chat history, it is part of the system instruction to address the recommendations outlined in the first evaluation. The model then changed its wording slighly to achieve higher sore. The improvement is marginal, given that the performance is already pretty good. These controls are sufficient in producing an improved summary. However, to get to a near perfect summary (summarization score ~ 0.95), one needs solid understanding of the article itself, in order to tailor the assessment questions/evaluation steps to a point where the summary accurately captures the core idea of the content. 

Please, do not forget to add your comments.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
